# Kanting - Video RAG System

A Video RAG (Retrieval Augmented Generation) system that:
1. Downloads YouTube videos
2. Extracts transcripts using Whisper (GPU)
3. Generates summaries using GPT-4o-mini
4. Stores in FAISS vector database
5. Answers questions with timestamped video links

---

## Setup & Installation

In [15]:
# Install dependencies
!pip install -q flask flask-cors openai faiss-cpu sentence-transformers
!pip install -q yt-dlp ffmpeg-python pyngrok python-dotenv
!pip install -q openai-whisper torch torchaudio

In [16]:
# Check GPU availability
import torch
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🎮 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")

🔧 PyTorch version: 2.9.0+cu126
🎮 CUDA available: True
🚀 GPU: Tesla T4


## Set Environment Variables

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "your-key"

# Set your ngrok auth token (get from https://dashboard.ngrok.com/get-started/your-authtoken)
NGROK_AUTH_TOKEN = "your-key"

# Enable GPU
os.environ["USE_GPU"] = "true"

print("✅ Environment configured")

✅ Environment configured


## Core Modules

In [18]:
# LLM Client (OpenAI GPT-4o-mini)

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"


def generate_summary(transcript: str) -> str:
    """Generate a brief summary of a video transcript."""
    text_only = []
    for line in transcript.strip().split('\n'):
        parts = line.rsplit(',', 2)
        if len(parts) >= 1:
            text_only.append(parts[0].strip())

    full_text = ' '.join(text_only)[:8000]

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You create concise video summaries."},
            {"role": "user", "content": f"Summarize in 2-3 sentences:\n\n{full_text}"}
        ],
        max_tokens=200
    )
    return response.choices[0].message.content.strip()


def generate_response(query: str, summary: str, context_chunks: list) -> str:
    """Generate answer using summary + relevant chunks."""
    chunks_text = "\n".join([f"[{c['start']:.1f}s]: {c['text']}" for c in context_chunks[:5]])

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer based on video content."},
            {"role": "user", "content": f"Summary: {summary}\n\nRelevant parts:\n{chunks_text}\n\nQuestion: {query}"}
        ],
        max_tokens=500
    )
    return response.choices[0].message.content.strip()


print("✅ LLM Client ready")

✅ LLM Client ready


In [19]:
# FAISS Vector Store

import json
import uuid
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer


class VectorStore:
    def __init__(self, device="cuda"):
        self.model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
        self.embedding_dim = 384
        self.index = faiss.IndexFlatIP(self.embedding_dim)
        self.metadata = {"videos": {}, "segments": []}

    def add_video(self, video_url, transcript, video_path=None, summary=None):
        video_id = f"video_{uuid.uuid4().hex[:8]}"
        self.metadata["videos"][video_id] = {
            "url": video_url, "path": video_path, "summary": summary
        }

        # Parse transcript
        segments = []
        for line in transcript.strip().split('\n'):
            parts = line.rsplit(',', 2)
            if len(parts) == 3:
                try:
                    segments.append({
                        "text": parts[0].strip(),
                        "start": float(parts[1]),
                        "end": float(parts[2]),
                        "video_id": video_id
                    })
                except: pass

        if segments:
            texts = [s["text"] for s in segments]
            embeddings = self.model.encode(texts, normalize_embeddings=True)
            self.index.add(np.array(embeddings).astype('float32'))
            self.metadata["segments"].extend(segments)

        print(f"✅ Added {len(segments)} segments from {video_id}")
        return video_id

    def search(self, query, top_k=2):
        if self.index.ntotal == 0:
            return []
        query_emb = self.model.encode([query], normalize_embeddings=True)
        sims, idxs = self.index.search(np.array(query_emb).astype('float32'), min(top_k, self.index.ntotal))
        return [{**self.metadata["segments"][i], "similarity": float(sims[0][j])}
                for j, i in enumerate(idxs[0]) if i < len(self.metadata["segments"])]

    def get_video_url(self, video_id):
        return self.metadata["videos"].get(video_id, {}).get("url")

    def get_video_summary(self, video_id):
        return self.metadata["videos"].get(video_id, {}).get("summary")

    def get_stats(self):
        return {"videos": len(self.metadata["videos"]), "segments": self.index.ntotal}


# Initialize
vector_store = VectorStore(device="cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Vector Store ready ({vector_store.get_stats()})")

✅ Vector Store ready ({'videos': 0, 'segments': 0})


In [20]:
# Video Downloader & Transcript Extraction

import whisper
import yt_dlp

# Load Whisper model (this takes a moment)
print("🎙️ Loading Whisper model...")
whisper_model = whisper.load_model("medium", device="cuda" if torch.cuda.is_available() else "cpu")
print("✅ Whisper ready")


def download_video(url, output_dir="videos"):
    """Download YouTube video."""
    os.makedirs(output_dir, exist_ok=True)
    video_name = f"video_{uuid.uuid4().hex[:8]}"

    ydl_opts = {
        'format': 'best[height<=720]',
        'outtmpl': f'{output_dir}/{video_name}.%(ext)s',
        'quiet': True
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        return ydl.prepare_filename(info)


def get_transcript(video_path):
    """Extract transcript using Whisper."""
    result = whisper_model.transcribe(video_path, verbose=False)
    lines = [f"{s['text'].strip()},{s['start']:.2f},{s['end']:.2f}" for s in result['segments']]
    return "\n".join(lines)

🎙️ Loading Whisper model...
✅ Whisper ready


## Flask API

In [34]:
from flask import Flask, request, jsonify
from flask_cors import CORS

app = Flask(__name__)
CORS(app)


@app.route('/store-transcript', methods=['POST'])
def store_transcript_endpoint():
    data = request.get_json()
    video_url = data.get('video_url')

    if not video_url:
        return jsonify({"error": "video_url required"}), 400

    try:
        # Download
        print(f"📥 Downloading: {video_url}")
        video_path = download_video(video_url)

        # Transcribe
        print("🎙️ Transcribing...")
        transcript = get_transcript(video_path)

        # Summarize
        print("📝 Generating summary...")
        summary = generate_summary(transcript)

        # Store
        video_id = vector_store.add_video(video_url, transcript, video_path, summary)

        return jsonify({"video_id": video_id, "summary": summary, "status": "success"}), 201

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route('/query', methods=['POST'])
def query_endpoint():
    data = request.get_json()
    query = data.get('query')

    if not query:
        return jsonify({"error": "query required"}), 400

    try:
        # Search
        segments = vector_store.search(query, top_k=3)

        if not segments:
            return jsonify({"answer": "No videos indexed yet.", "video_url": None})

        # Get best video
        video_groups = {}
        for seg in segments:
            vid = seg["video_id"]
            video_groups.setdefault(vid, []).append(seg)

        best_video_id = max(video_groups, key=lambda v: sum(s["similarity"] for s in video_groups[v]))
        best_segments = video_groups[best_video_id]

        # Get video info
        video_url = vector_store.get_video_url(best_video_id)
        summary = vector_store.get_video_summary(best_video_id)

        # Timestamp
        timestamp = int(min(s["start"] for s in best_segments))
        timestamped_url = f"{video_url}&t={timestamp}" if "?" in video_url else f"{video_url}?t={timestamp}"

        # Generate answer
        answer = generate_response(query, summary or "", best_segments)

        return jsonify({
            "answer": answer,
            "video_url": timestamped_url,
            "timestamp": timestamp,
            "segments_used": len(best_segments)
        })

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route('/stats', methods=['GET'])
def stats_endpoint():
    return jsonify(vector_store.get_stats())


@app.route('/health', methods=['GET'])
def health_check():
    return jsonify({"status": "healthy"})


print("✅ Flask app ready")

✅ Flask app ready


## Start Server with ngrok

In [35]:
from pyngrok import ngrok
import threading

# Set ngrok auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start Flask in a thread
def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask)
flask_thread.daemon = True
flask_thread.start()

# Create ngrok tunnel
public_url = ngrok.connect(5000)
print("="*60)
print(f"🌐 PUBLIC URL: {public_url}")
print("="*60)
print("\nEndpoints:")
print(f"  POST {public_url}/store-transcript")
print(f"  POST {public_url}/query")
print(f"  GET  {public_url}/stats")
print(f"  GET  {public_url}/health")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


🌐 PUBLIC URL: NgrokTunnel: "https://knox-nondepreciatory-lobularly.ngrok-free.dev" -> "http://localhost:5000"

Endpoints:
  POST NgrokTunnel: "https://knox-nondepreciatory-lobularly.ngrok-free.dev" -> "http://localhost:5000"/store-transcript
  POST NgrokTunnel: "https://knox-nondepreciatory-lobularly.ngrok-free.dev" -> "http://localhost:5000"/query
  GET  NgrokTunnel: "https://knox-nondepreciatory-lobularly.ngrok-free.dev" -> "http://localhost:5000"/stats
  GET  NgrokTunnel: "https://knox-nondepreciatory-lobularly.ngrok-free.dev" -> "http://localhost:5000"/health


## Test the API

In [36]:
import requests

# Test health
new_url = public_url.data['public_url']
response = requests.get(f"{new_url}/health")
print(f"Health: {response.json()}")

INFO:werkzeug:127.0.0.1 - - [31/Dec/2025 00:26:54] "GET /health HTTP/1.1" 200 -


Health: {'status': 'healthy'}


In [37]:
# Add a video (replace with your YouTube URL)
VIDEO_URL = "https://www.youtube.com/watch?v=ZXiruGOCn9s"  # <-- Replace with your video

response = requests.post(
    f"{new_url}/store-transcript",
    json={"video_url": VIDEO_URL}
)
print(response.json())

📥 Downloading: https://www.youtube.com/watch?v=ZXiruGOCn9s


🎙️ Transcribing...
Detected language: English


100%|██████████| 35006/35006 [00:35<00:00, 991.84frames/s]


📝 Generating summary...


INFO:werkzeug:127.0.0.1 - - [31/Dec/2025 00:27:43] "POST /store-transcript HTTP/1.1" 201 -


✅ Added 74 segments from video_1495e69d
{'status': 'success', 'summary': "This video explains the capabilities of GPT-3, a Generative Pre-trained Transformer model, which can create human-like text, including jokes and translations. It highlights the transformer architecture's use of encoders and decoders, the attention mechanism that improves context understanding, and its advantages over traditional models like RNNs by processing data in parallel. Beyond language tasks, transformers are versatile tools for document summarization, content generation, and even complex tasks like image processing and game playing.", 'video_id': 'video_1495e69d'}


In [38]:
# Ask a question
response = requests.post(
    f"{new_url}/query",
    json={"query": "what are encoders"}
)
result = response.json()

print(f"Answer: {result.get('answer')}")
print(f"Video URL: {result.get('video_url')}")

INFO:werkzeug:127.0.0.1 - - [31/Dec/2025 00:27:49] "POST /query HTTP/1.1" 200 -


Answer: Encoders are a key component of Transformer models, like GPT-3, that process input sequences to create representations of the data. They generate encodings that define which parts of the input sequence are relevant, allowing the model to understand context and relationships between different elements of the input. The encoder works on the input sequence and passes these encodings to the next encoder layer, enabling the model to build a deeper understanding of the data through multiple layers of processing. 

In summary, encoders help to transform input data into a format that can be effectively used for tasks such as language translation, document summarization, and more, by capturing important context and relationships.
Video URL: https://www.youtube.com/watch?v=ZXiruGOCn9s&t=185


In [26]:
# Check stats
response = requests.get(f"{new_url}/stats")
print(f"Stats: {response.json()}")

INFO:werkzeug:127.0.0.1 - - [31/Dec/2025 00:22:30] "GET /stats HTTP/1.1" 200 -


Stats: {'segments': 74, 'videos': 1}


---
## Done!

Your Video RAG system is now running. Use the public URL above to:
1. Add videos via `/store-transcript`
2. Ask questions via `/query`
3. Check stats via `/stats`